# 10N-AI: Asistente Inteligente de Machine Learning
### Proyecto I - Inteligencia Artificial | ITCR San Carlos
### Modelo 04: Predicción del Precio del Aguacate (Random Forest vs. Red Neuronal MLP)
**Estudiantes:** Carlos Arce & Joshua Picado  
**Profesor:** Ing. Efrén Jiménez Delgado

# Análisis del Problema

### Formulación del Problema
El mercado del aguacate Hass en Estados Unidos presenta una volatilidad de precios marcada por la estacionalidad de las cosechas, los volúmenes comercializados y la fuerte segmentación entre el cultivo convencional y el orgánico. El objetivo de este modelo es predecir con exactitud el **precio medio por unidad (`AveragePrice`)** en dólares ($USD) a partir de variables de oferta agregada (`Total Volume`), desglose de calibres (PLU 4046, 4225, 4770, empaques en bolsas), método de cultivo y factor temporal (mes del año).

Este modelo funcionará como una habilidad de analítica comercial en el asistente **10N-AI**, permitiendo que un usuario consulte: *"10N-AI, estimá el precio de una caja de aguacates orgánicos para el mes de septiembre con volumen medio en California"*.

### Selección y Comparación de Modelos Vistos en Clase
Para este problema de **regresión continua**, se propone una comparativa directa entre dos familias algorítmicas fundamentales estudiadas en el curso:
1. **Bosque Aleatorio / Random Forest Regressor (Clase 3 - Métodos de Ensamble):** Arquitectura no paramétrica basada en agregación de árboles de decisión independientes con selección aleatoria de variables (*Feature Bagging*).
2. **Red Neuronal Artificial / Multi-Layer Perceptron Regressor - MLP (Clase 6 - Deep Learning y Redes Neuronales):** Arquitectura de aprendizaje profundo compuesta por múltiples capas densas ocultas con funciones de activación no lineales (*ReLU*) y optimización por gradiente descendente estocástico (*Adam*).

#### Justificación de la Competencia de Arquitecturas:
* **Capacidad de Modelado No Lineal:** Tanto los ensambles como las redes multicapa poseen capacidad de aproximación universal ante superficies de respuesta complejas.
* **Comportamiento en Datos Tabulares:** Las redes neuronales profundas sobresalen al mapear representaciones continuas densas, mientras que los árboles son reconocidos por su eficacia al particionar datos tabulares con variables categóricas de región y tipo.
* **Criterio de Selección para 10N-AI:** Ambas arquitecturas se entrenan bajo validación cruzada y se evalúan bajo las mismas métricas ($R^2$, MAE, RMSE). El modelo con mejor rendimiento matemático objetivo será seleccionado y exportado automáticamente para su uso en la API REST del asistente.

In [1]:
# @title Importación de Librerías y Configuración Global { display-mode: "form" }
import os
import sys
import warnings
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

warnings.filterwarnings('ignore')
print("Librerías importadas exitosamente para 10N-AI.")

Librerías importadas exitosamente para 10N-AI.


In [2]:
# @title Conexión a Google Drive y Carga del Dataset { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')

ruta_dataset = '/content/drive/MyDrive/Datasets/avocado.csv'

# Verificación de rutas alternativas
if not os.path.exists(ruta_dataset):
    rutas_alt = [
        '/content/drive/MyDrive/Datasets/Avocado.csv',
        '/content/drive/MyDrive/Datasets/avocado'
    ]
    for r in rutas_alt:
        if os.path.exists(r):
            ruta_dataset = r
            break
    else:
        raise FileNotFoundError(
            f"No se encontró el archivo en: {ruta_dataset}. "
            "Asegúrate de haber montado Google Drive y de tener el archivo en la carpeta 'Datasets'."
        )

print(f"Cargando dataset desde Google Drive: {ruta_dataset}")
df = pd.read_csv(ruta_dataset)
print(f"Dataset cargado con éxito. Dimensiones: {df.shape[0]} filas y {df.shape[1]} columnas.")
display(df.head())

Mounted at /content/drive
Cargando dataset desde Google Drive: /content/drive/MyDrive/Datasets/avocado.csv
Dataset cargado con éxito. Dimensiones: 18249 filas y 14 columnas.


,Unnamed: 0,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region
0,0,2015-12-27,1.33,64236.62,1036.74,54454.85,48.16,8696.87,8603.62,93.25,0.0,conventional,2015,Albany
1,1,2015-12-20,1.35,54876.98,674.28,44638.81,58.33,9505.56,9408.07,97.49,0.0,conventional,2015,Albany
2,2,2015-12-13,0.93,118220.22,794.70,109149.67,130.50,8145.35,8042.21,103.14,0.0,conventional,2015,Albany
3,3,2015-12-06,1.08,78992.15,1132.00,71976.41,72.58,5811.16,5677.40,133.76,0.0,conventional,2015,Albany
4,4,2015-11-29,1.28,51039.60,941.48,43838.39,75.78,6183.95,5986.26,197.69,0.0,conventional,2015,Albany


# Entendimiento de los Datos

### Diccionario de Variables
* **`Date` (Fecha):** Fecha del registro semanal.
* **`AveragePrice` (Numérica continua - Target):** Precio promedio de un aguacate en dólares ($USD).
* **`Total Volume` (Numérica continua):** Número total de aguacates vendidos.
* **`4046` (Numérica continua):** Aguacates pequeños/medianos vendidos con código PLU 4046.
* **`4225` (Numérica continua):** Aguacates grandes vendidos con código PLU 4225.
* **`4770` (Numérica continua):** Aguacates extra grandes vendidos con código PLU 4770.
* **`Total Bags`, `Small Bags`, `Large Bags`, `XLarge Bags` (Numéricas):** Desglose de bolsas vendidas.
* **`type` (Categórica nominal):** Tipo de cultivo (`conventional` o `organic`).
* **`year` (Numérica discreta):** Año de la transacción (2015 - 2018).
* **`region` (Categórica nominal):** Región o ciudad del mercado en EE.UU.

In [3]:
# @title Inspección de Estructura, Tipos de Datos y Detección de Nulos { display-mode: "form" }
print("=== INFORMACIÓN GENERAL DEL DATASET ===")
print(df.info())
print("\n" + "="*60 + "\n")

print("=== CONTEO DE VALORES NULOS POR COLUMNA ===")
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else "No existen valores nulos en el dataset.")
print("\n" + "="*60 + "\n")

# Eliminación de columnas residuales
cols_eliminar = [c for c in df.columns if 'unnamed' in c.lower()]
if cols_eliminar:
    df.drop(columns=cols_eliminar, inplace=True)
    print(f"Columnas residuales eliminadas: {cols_eliminar}")
    print("\n" + "="*60 + "\n")

print("=== REGISTROS DUPLICADOS ===")
duplicados = df.duplicated().sum()
print(f"Cantidad de filas duplicadas: {duplicados}")
if duplicados > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Duplicados eliminados. Nuevas dimensiones: {df.shape}")
print("\n" + "="*60 + "\n")

print("=== RESUMEN ESTADÍSTICO DE VARIABLES NUMÉRICAS ===")
display(df.describe().T.round(2))

=== INFORMACIÓN GENERAL DEL DATASET ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18249 entries, 0 to 18248
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Unnamed: 0    18249 non-null  int64  
 1   Date          18249 non-null  object 
 2   AveragePrice  18249 non-null  float64
 3   Total Volume  18249 non-null  float64
 4   4046          18249 non-null  float64
 5   4225          18249 non-null  float64
 6   4770          18249 non-null  float64
 7   Total Bags    18249 non-null  float64
 8   Small Bags    18249 non-null  float64
 9   Large Bags    18249 non-null  float64
 10  XLarge Bags   18249 non-null  float64
 11  type          18249 non-null  object 
 12  year          18249 non-null  int64  
 13  region        18249 non-null  object 
dtypes: float64(9), int64(2), object(3)
memory usage: 1.9+ MB
None


=== CONTEO DE VALORES NULOS POR COLUMNA ===
No existen valores nulos en el dataset.


Columnas

,count,mean,std,min,25%,50%,75%,max
AveragePrice,18249.0,1.41,0.40,0.44,1.10,1.37,1.66,3.25
Total Volume,18249.0,850644.01,3453545.36,84.56,10838.58,107376.76,432962.29,62505646.52
4046,18249.0,293008.42,1264989.08,0.00,854.07,8645.30,111020.20,22743616.17
4225,18249.0,295154.57,1204120.40,0.00,3008.78,29061.02,150206.86,20470572.61
4770,18249.0,22839.74,107464.07,0.00,0.00,184.99,6243.42,2546439.11
Total Bags,18249.0,239639.20,986242.40,0.00,5088.64,39743.83,110783.37,19373134.37
Small Bags,18249.0,182194.69,746178.51,0.00,2849.42,26362.82,83337.67,13384586.80
Large Bags,18249.0,54338.09,243965.96,0.00,127.47,2647.71,22029.25,5719096.61
XLarge Bags,18249.0,3106.43,17692.89,0.00,0.00,0.00,132.50,551693.65
year,18249.0,2016.15,0.94,2015.00,2015.00,2016.00,2017.00,2018.00


### Hallazgos Principales del Entendimiento de Datos
1. **Completitud:** Las 18,249 instancias están completas sin valores nulos en ninguna columna.
2. **Remoción de Índice Residual:** Se remueve la columna `Unnamed: 0` proveniente de la exportación inicial en disco.
3. **Filtro de TotalUS:** La variable `region` contiene `'TotalUS'` como consolidado nacional, el cual se descarta para evitar duplicidad de volúmenes con las regiones individuales.

# Exploración de los Datos

In [4]:
# @title Distribución Interactiva del Precio Promedio (AveragePrice) { display-mode: "form" }
fig_hist = px.histogram(
    df,
    x='AveragePrice',
    nbins=40,
    marginal='box',
    title='<b>Distribución del Precio Promedio del Aguacate ($USD) con Boxplot Marginal</b>',
    labels={'AveragePrice': 'Precio Promedio ($USD)'},
    color_discrete_sequence=['#16a34a']
)

fig_hist.update_layout(
    template='plotly_white',
    xaxis_title='Precio Promedio ($USD)',
    yaxis_title='Frecuencia'
)
fig_hist.show()

# Análisis
print("--- ANÁLISIS DE LA VARIABLE OBJETIVO (AVERAGEPRICE) ---")
print(f"• Mediana vs Media: La mediana se ubica en ${df['AveragePrice'].median():.2f} USD y la media en ${df['AveragePrice'].mean():.2f} USD.")
print("• Rango Modal: La mayor concentración de transacciones ocurre entre $1.10 y $1.65 USD.")
print("• Dispersión Superior: Precios superiores a $2.50 corresponden predominantemente al segmento orgánico.")

--- ANÁLISIS DE LA VARIABLE OBJETIVO (AVERAGEPRICE) ---
• Mediana vs Media: La mediana se ubica en $1.37 USD y la media en $1.41 USD.
• Rango Modal: La mayor concentración de transacciones ocurre entre $1.10 y $1.65 USD.
• Dispersión Superior: Precios superiores a $2.50 corresponden predominantemente al segmento orgánico.


In [5]:
# @title Comparativa de Precios según Tipo de Cultivo (Orgánico vs Convencional) { display-mode: "form" }
fig_type = px.box(
    df,
    x='type',
    y='AveragePrice',
    color='type',
    title='<b>Distribución de Precios según Tipo de Cultivo</b>',
    labels={'type': 'Tipo de Aguacate', 'AveragePrice': 'Precio Promedio ($USD)'},
    color_discrete_map={'conventional': '#16a34a', 'organic': '#d97706'}
)

fig_type.update_layout(
    template='plotly_white',
    xaxis_title='Tipo de Cultivo',
    yaxis_title='Precio Promedio ($USD)'
)
fig_type.show()

# Análisis
print("--- ANÁLISIS: TIPO DE CULTIVO Y PRECIO ---")
print("• Brecha Orgánica: El aguacate orgánico tiene una mediana de precio (~$1.65 USD) sensiblemente superior al convencional (~$1.15 USD).")
print("• Dispersión en Orgánicos: El cultivo orgánico presenta una variabilidad mayor de precios debido a menores rendimientos por hectárea.")

--- ANÁLISIS: TIPO DE CULTIVO Y PRECIO ---
• Brecha Orgánica: El aguacate orgánico tiene una mediana de precio (~$1.65 USD) sensiblemente superior al convencional (~$1.15 USD).
• Dispersión en Orgánicos: El cultivo orgánico presenta una variabilidad mayor de precios debido a menores rendimientos por hectárea.


In [6]:
# @title Estacionalidad Mensual del Precio del Aguacate { display-mode: "form" }
df_temp = df.copy()
df_temp['Date'] = pd.to_datetime(df_temp['Date'])
df_temp['Month'] = df_temp['Date'].dt.month

promedio_mensual = df_temp.groupby(['Month', 'type'])['AveragePrice'].mean().reset_index()

fig_mes = px.line(
    promedio_mensual,
    x='Month',
    y='AveragePrice',
    color='type',
    markers=True,
    title='<b>Tendencia Mensual del Precio Promedio según Tipo de Aguacate</b>',
    labels={'Month': 'Mes del Año', 'AveragePrice': 'Precio Medio ($USD)', 'type': 'Tipo'},
    color_discrete_map={'conventional': '#16a34a', 'organic': '#d97706'}
)

fig_mes.update_layout(
    template='plotly_white',
    xaxis=dict(tickmode='linear', tick0=1, dtick=1),
    xaxis_title='Mes (1 = Enero, 12 = Diciembre)',
    yaxis_title='Precio Promedio ($USD)'
)
fig_mes.show()

# Análisis
print("--- ANÁLISIS: ESTACIONALIDAD MENSUAL ---")
print("• Pico en Otoño: Ambos tipos muestran su punto de precio más alto entre agosto y octubre (meses 8 a 10) por menor cosecha en California.")
print("• Mínimos a Inicio de Año: Febrero y marzo registran los precios más bajos impulsados por la temporada alta de importación mexicana.")

--- ANÁLISIS: ESTACIONALIDAD MENSUAL ---
• Pico en Otoño: Ambos tipos muestran su punto de precio más alto entre agosto y octubre (meses 8 a 10) por menor cosecha en California.
• Mínimos a Inicio de Año: Febrero y marzo registran los precios más bajos impulsados por la temporada alta de importación mexicana.


In [7]:
# @title Matriz de Correlación Interactiva de Volúmenes y Precios { display-mode: "form" }
df_corr = df.copy()
df_corr['is_organic'] = (df_corr['type'] == 'organic').astype(int)
cols_num = ['AveragePrice', 'Total Volume', '4046', '4225', '4770', 'Total Bags', 'is_organic']
matriz_corr = df_corr[cols_num].corr()

fig_heat = px.imshow(
    matriz_corr,
    text_auto='.2f',
    color_continuous_scale='Greens',
    zmin=-1.0,
    zmax=1.0,
    title='<b>Matriz de Correlación Interactiva de Variables Numéricas</b>'
)

fig_heat.update_layout(
    template='plotly_white',
    width=750,
    height=600
)
fig_heat.show()

# Análisis
print("--- ANÁLISIS DE LA MATRIZ DE CORRELACIÓN ---")
print("• Total Volume vs AveragePrice (r = -0.19): Relación inversa clásica de oferta; mayores volúmenes empujan el precio a la baja.")
print("• is_organic vs AveragePrice (r = +0.62): Fuerte correlación positiva; ser orgánico es el predictor individual más potente del precio.")
print("• Alta Correlación entre Calibres: Las variedades 4046, 4225 y Total Bags muestran colinealidad con el volumen total.")

--- ANÁLISIS DE LA MATRIZ DE CORRELACIÓN ---
• Total Volume vs AveragePrice (r = -0.19): Relación inversa clásica de oferta; mayores volúmenes empujan el precio a la baja.
• is_organic vs AveragePrice (r = +0.62): Fuerte correlación positiva; ser orgánico es el predictor individual más potente del precio.
• Alta Correlación entre Calibres: Las variedades 4046, 4225 y Total Bags muestran colinealidad con el volumen total.


# Modelo de machine learning

### Ingeniería de Características y Preparación de Datos
1. **Filtro de Mercados Regionales:** Se excluyen los registros de `'TotalUS'`.
2. **Ingeniería Temporal:** Extracción del mes (`Month`) a partir de la fecha.
3. **Codificación:** Binarización de `type` (`is_organic`: 1 si es orgánico, 0 convencional) y One-Hot Encoding de `region` con `drop_first=True`.
4. **Partición y Escalamiento:** División 80% Train y 20% Test. Se ajusta un `StandardScaler` sobre los datos de entrenamiento para alimentar la Red Neuronal Artificial (imprescindible para la convergencia del algoritmo Adam).

In [8]:
# @title Preparación de Datos, Feature Engineering y Escalamiento { display-mode: "form" }
df_ml = df[df['region'] != 'TotalUS'].copy()

# 1. Extracción de mes
df_ml['Date'] = pd.to_datetime(df_ml['Date'])
df_ml['Month'] = df_ml['Date'].dt.month
df_ml.drop('Date', axis=1, inplace=True)

# 2. Binarización de type
df_ml['is_organic'] = (df_ml['type'] == 'organic').astype(int)
df_ml.drop('type', axis=1, inplace=True)

# 3. One-Hot Encoding de region
df_ml = pd.get_dummies(df_ml, columns=['region'], drop_first=True)

X = df_ml.drop('AveragePrice', axis=1)
y = df_ml['AveragePrice']

columnas_predictoras = list(X.columns)

# 4. Partición Train / Test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# 5. Escalamiento estándar (requerido por la Red Neuronal)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dimensiones de entrenamiento (X_train): {X_train.shape}")
print(f"Dimensiones de prueba (X_test): {X_test.shape}")
print(f"Total de variables predictoras: {len(columnas_predictoras)}")

Dimensiones de entrenamiento (X_train): (14328, 63)
Dimensiones de prueba (X_test): (3583, 63)
Total de variables predictoras: 63


In [9]:
# @title Entrenamiento y Optimización de Random Forest Regressor  { display-mode: "form" }
print("Entrenando y optimizando Bosque Aleatorio (Random Forest)...")
espacio_rf = {
    'n_estimators': [100, 150],
    'max_depth': [16, 20, None],
    'min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=espacio_rf,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='r2',
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)
modelo_rf = grid_rf.best_estimator_

print(f"Mejor R² en CV (Random Forest): {grid_rf.best_score_:.4f}")
print("Hiperparámetros óptimos RF:", grid_rf.best_params_)

Entrenando y optimizando Bosque Aleatorio (Random Forest)...
Mejor R² en CV (Random Forest): 0.8937
Hiperparámetros óptimos RF: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 150}


In [10]:
# @title Entrenamiento de la Red Neuronal Artificial - MLPRegressor  { display-mode: "form" }
print("Entrenando Red Neuronal Artificial (Multi-Layer Perceptron)...")
modelo_mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate_init=0.01,
    max_iter=300,
    early_stopping=True,
    random_state=42
)

# Las redes neuronales se entrenan obligatoriamente sobre los datos escalados
modelo_mlp.fit(X_train_scaled, y_train)

# Validación cruzada de 5 pliegues para la Red Neuronal
cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_mlp_cv = cross_val_score(modelo_mlp, X_train_scaled, y_train, cv=cv_kfold, scoring='r2')

print(f"R² promedio en CV (Red Neuronal MLP): {scores_mlp_cv.mean():.4f} (+/- {scores_mlp_cv.std():.4f})")
print(f"Iteraciones completadas antes de convergencia: {modelo_mlp.n_iter_}")

Entrenando Red Neuronal Artificial (Multi-Layer Perceptron)...
R² promedio en CV (Red Neuronal MLP): 0.8443 (+/- 0.0106)
Iteraciones completadas antes de convergencia: 83


In [12]:
# @title Importancia Relativa de las Características (Random Forest) { display-mode: "form" }
df_importancias = pd.DataFrame({
    'Variable': columnas_predictoras,
    'Importancia': modelo_rf.feature_importances_
}).sort_values('Importancia', ascending=True).tail(12)

fig_imp = px.bar(
    df_importancias,
    x='Importancia',
    y='Variable',
    orientation='h',
    title='<b>Top 12 Variables con Mayor Importancia en la Predicción del Precio (Gini)</b>',
    labels={'Importancia': 'Importancia (Gini)', 'Variable': 'Característica'},
    color_discrete_sequence=['#2E86C1']  # Color sólido consistente con los modelos anteriores
)

fig_imp.update_layout(
    template='plotly_white',
    xaxis_title='Importancia Relativa',
    yaxis_title='Variable Predictora'
)
fig_imp.show()

# Análisis
print("--- ANÁLISIS DE IMPORTANCIA DE VARIABLES ---")
print("• is_organic: Es el factor determinante dominante; el sobrecosto orgánico rige la mayor parte de las divisiones.")
print("• Total Volume y 4046: Aportan la señal de oferta de mercado requerida.")
print("• Month: Captura la estacionalidad agronómica del producto.")

--- ANÁLISIS DE IMPORTANCIA DE VARIABLES ---
• is_organic: Es el factor determinante dominante; el sobrecosto orgánico rige la mayor parte de las divisiones.
• Total Volume y 4046: Aportan la señal de oferta de mercado requerida.
• Month: Captura la estacionalidad agronómica del producto.


# Evaluación

In [14]:
# @title Duelo de Modelos: Evaluación Comparativa en Conjunto de Prueba { display-mode: "form" }
# Predicciones
y_pred_rf = modelo_rf.predict(X_test)
y_pred_mlp = modelo_mlp.predict(X_test_scaled)

# Métricas Random Forest
r2_rf = r2_score(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

# Métricas Red Neuronal
r2_mlp = r2_score(y_test, y_pred_mlp)
mae_mlp = mean_absolute_error(y_test, y_pred_mlp)
rmse_mlp = np.sqrt(mean_squared_error(y_test, y_pred_mlp))

comparativa = pd.DataFrame({
    'Métrica': ['R² Score (Bondad de Ajuste)', 'MAE (Error Absoluto Medio)', 'RMSE (Raíz ECM)'],
    'Random Forest (Clase 3)': [f"{r2_rf:.4f}", f"${mae_rf:.4f} USD", f"${rmse_rf:.4f} USD"],
    'Red Neuronal MLP (Clase 6)': [f"{r2_mlp:.4f}", f"${mae_mlp:.4f} USD", f"${rmse_mlp:.4f} USD"]
})

display(comparativa)

# @title Duelo de Modelos: Evaluación Comparativa en Conjunto de Prueba { display-mode: "form" }
# Predicciones
y_pred_rf = modelo_rf.predict(X_test)
y_pred_mlp = modelo_mlp.predict(X_test_scaled)

# Métricas Random Forest
r2_rf = r2_score(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

# Métricas Red Neuronal
r2_mlp = r2_score(y_test, y_pred_mlp)
mae_mlp = mean_absolute_error(y_test, y_pred_mlp)
rmse_mlp = np.sqrt(mean_squared_error(y_test, y_pred_mlp))

comparativa = pd.DataFrame({
    'Métrica': ['R² Score (Bondad de Ajuste)', 'MAE (Error Absoluto Medio)', 'RMSE (Raíz ECM)'],
    'Random Forest (Clase 3)': [f"{r2_rf:.4f}", f"${mae_rf:.4f} USD", f"${rmse_rf:.4f} USD"],
    'Red Neuronal MLP (Clase 6)': [f"{r2_mlp:.4f}", f"${mae_mlp:.4f} USD", f"${rmse_mlp:.4f} USD"]
})

display(comparativa)

# Análisis
print("--- ANÁLISIS COMPARATIVO DE RENDIMIENTO ---")
if r2_rf >= r2_mlp:
    print(f"• Ganador: Bosque Aleatorio (Random Forest) con un R² de {r2_rf*100:.2f}% frente al {r2_mlp*100:.2f}% de la Red Neuronal.")
    print(f"• El error promedio (MAE) de Random Forest fue de solo ${mae_rf:.2f} USD frente a ${mae_mlp:.2f} USD de la Red Neuronal.")
else:
    print(f"• Ganador: Red Neuronal Artificial con un R² de {r2_mlp*100:.2f}% frente al {r2_rf*100:.2f}% del Bosque Aleatorio.")
    print(f"• El error promedio (MAE) de la Red Neuronal fue de ${mae_mlp:.2f} USD.")

,Métrica,Random Forest (Clase 3),Red Neuronal MLP (Clase 6)
0,R² Score (Bondad de Ajuste),0.9089,0.8578
1,MAE (Error Absoluto Medio),$0.0848 USD,$0.1093 USD
2,RMSE (Raíz ECM),$0.1227 USD,$0.1533 USD


,Métrica,Random Forest (Clase 3),Red Neuronal MLP (Clase 6)
0,R² Score (Bondad de Ajuste),0.9089,0.8578
1,MAE (Error Absoluto Medio),$0.0848 USD,$0.1093 USD
2,RMSE (Raíz ECM),$0.1227 USD,$0.1533 USD


--- ANÁLISIS COMPARATIVO DE RENDIMIENTO ---
• Ganador: Bosque Aleatorio (Random Forest) con un R² de 90.89% frente al 85.78% de la Red Neuronal.
• El error promedio (MAE) de Random Forest fue de solo $0.08 USD frente a $0.11 USD de la Red Neuronal.


In [15]:
# @title Gráfico Interactivo de Valores Reales vs. Predichos (Modelo Ganador) { display-mode: "form" }
# Determinamos el vector del ganador
pred_ganador = y_pred_rf if r2_rf >= r2_mlp else y_pred_mlp
nombre_ganador_str = "Random Forest Regressor" if r2_rf >= r2_mlp else "Red Neuronal Artificial (MLP)"

df_eval = pd.DataFrame({
    'Real': y_test,
    'Predicho': pred_ganador,
    'Residuo': y_test - pred_ganador
})

df_eval_sample = df_eval.sample(min(2000, len(df_eval)), random_state=42)

fig_pred = px.scatter(
    df_eval_sample,
    x='Real',
    y='Predicho',
    hover_data=['Residuo'],
    title=f'<b>Valores Reales vs. Predichos en Prueba: {nombre_ganador_str}</b>',
    labels={'Real': 'Precio Real ($USD)', 'Predicho': 'Precio Predicho ($USD)'},
    color_discrete_sequence=['#16a34a']
)

min_val = min(df_eval_sample['Real'].min(), df_eval_sample['Predicho'].min())
max_val = max(df_eval_sample['Real'].max(), df_eval_sample['Predicho'].max())

fig_pred.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    name='Ajuste Perfecto (y = x)',
    line=dict(color='#ef4444', dash='dash', width=2)
))

fig_pred.update_layout(
    template='plotly_white',
    xaxis_title='Precio Real ($USD)',
    yaxis_title='Precio Predicho ($USD)'
)
fig_pred.show()

# Análisis
print("--- ANÁLISIS DE DISPERSIÓN DE PREDICCIÓN ---")
print("• Ajuste Lineal: La concentración de puntos alrededor de la diagonal (y = x) confirma alta precisión.")
print(f"• Estabilidad: El modelo mantiene consistencia tanto en precios bajos ($0.80) como en valores premium ($2.50).")

--- ANÁLISIS DE DISPERSIÓN DE PREDICCIÓN ---
• Ajuste Lineal: La concentración de puntos alrededor de la diagonal (y = x) confirma alta precisión.
• Estabilidad: El modelo mantiene consistencia tanto en precios bajos ($0.80) como en valores premium ($2.50).


In [16]:
# @title Distribución Interactiva de Residuos (Errores del Modelo Ganador) { display-mode: "form" }
fig_res = px.histogram(
    df_eval,
    x='Residuo',
    nbins=40,
    marginal='box',
    title=f'<b>Distribución de Errores Residuales: {nombre_ganador_str}</b>',
    labels={'Residuo': 'Error Residual ($USD)'},
    color_discrete_sequence=['#f59e0b']
)

fig_res.update_layout(
    template='plotly_white',
    xaxis_title='Error Residual ($USD)',
    yaxis_title='Frecuencia'
)
fig_res.show()

# Análisis
print("--- DIAGNÓSTICO DE RESIDUOS ---")
print(f"• Media de Residuos: {df_eval['Residuo'].mean():.4f} (distribución centrada en cero, libre de sesgo sistemático).")
print(f"• Desviación de Errores: ${df_eval['Residuo'].std():.4f} USD.")

--- DIAGNÓSTICO DE RESIDUOS ---
• Media de Residuos: -0.0013 (distribución centrada en cero, libre de sesgo sistemático).
• Desviación de Errores: $0.1227 USD.


# Resultados

### Selección Automática, Simulador y Exportación para 10N-AI
La siguiente celda evalúa automáticamente cuál modelo obtuvo el mayor coeficiente $R^2$, define la función de inferencia interactiva para **10N-AI** y genera el artefacto binario `.joblib` empaquetado para la API REST.

In [17]:
# @title Selección Automática del Ganador y Simulador para 10N-AI { display-mode: "form" }
# Selección objetiva por métricas
if r2_rf >= r2_mlp:
    modelo_final = modelo_rf
    nombre_ganador = "Random Forest Regressor"
    usar_scaler = False
    r2_ganador = r2_rf
    mae_ganador = mae_rf
    rmse_ganador = rmse_rf
else:
    modelo_final = modelo_mlp
    nombre_ganador = "Red Neuronal Artificial (MLPRegressor)"
    usar_scaler = True
    r2_ganador = r2_mlp
    mae_ganador = mae_mlp
    rmse_ganador = rmse_mlp

print("="*65)
print(f"🏆 MODELO GANADOR SELECCIONADO PARA 10N-AI: {nombre_ganador}")
print(f"• R² Score: {r2_ganador:.4f} | MAE: ${mae_ganador:.4f} USD | RMSE: ${rmse_ganador:.4f} USD")
print("="*65)

def cotizar_aguacate_10n(total_volume=50000.0, plu_4046=15000.0, plu_4225=20000.0, plu_4770=1000.0,
                         total_bags=14000.0, small_bags=10000.0, large_bags=3800.0, xlarge_bags=200.0,
                         year=2026, month=9, is_organic=1, region='California'):
    """
    Función de inferencia consumible por el asistente 10N-AI.
    """
    fila = {col: 0.0 for col in columnas_predictoras}
    fila['Total Volume'] = total_volume
    fila['4046'] = plu_4046
    fila['4225'] = plu_4225
    fila['4770'] = plu_4770
    fila['Total Bags'] = total_bags
    fila['Small Bags'] = small_bags
    fila['Large Bags'] = large_bags
    fila['XLarge Bags'] = xlarge_bags
    fila['year'] = year
    fila['Month'] = month
    fila['is_organic'] = is_organic

    col_region = f'region_{region}'
    if col_region in fila:
        fila[col_region] = 1.0

    entrada_df = pd.DataFrame([fila])[columnas_predictoras]

    if usar_scaler:
        entrada_proc = scaler.transform(entrada_df)
        precio_predicho = modelo_final.predict(entrada_proc)[0]
    else:
        precio_predicho = modelo_final.predict(entrada_df)[0]

    cultivo_str = "Orgánico" if is_organic == 1 else "Convencional"
    print("="*65)
    print(f"🤖 [10N-AI ASISTENTE]: Cotización de Aguacate Hass:")
    print(f"• Motor de Inferencia: {nombre_ganador}")
    print(f"• Cultivo: {cultivo_str} | Región: {region} | Mes: {month}/{year}")
    print(f"• Volumen Proyectado: {total_volume:,} unidades")
    print(f"🥑 PRECIO ESTIMADO POR AGUACATE: ${precio_predicho:.2f} USD")
    print("="*65)
    return precio_predicho

# Prueba de simulación
_ = cotizar_aguacate_10n(total_volume=75000, month=9, is_organic=1, region='California')

🏆 MODELO GANADOR SELECCIONADO PARA 10N-AI: Random Forest Regressor
• R² Score: 0.9089 | MAE: $0.0848 USD | RMSE: $0.1227 USD
🤖 [10N-AI ASISTENTE]: Cotización de Aguacate Hass:
• Motor de Inferencia: Random Forest Regressor
• Cultivo: Orgánico | Región: California | Mes: 9/2026
• Volumen Proyectado: 75,000 unidades
🥑 PRECIO ESTIMADO POR AGUACATE: $2.08 USD


In [19]:
# @title Exportación del Modelo Ganador para la API REST { display-mode: "form" }
from google.colab import files
import joblib

os.makedirs('/content/models', exist_ok=True)
ruta_export = '/content/models/modelo_04_aguacate.joblib'

paquete_modelo = {
    'model': modelo_final,
    'nombre': nombre_ganador,
    'scaler': scaler if usar_scaler else None,
    'features': columnas_predictoras,
    'metrics': {
        'r2': float(r2_ganador),
        'mae': float(mae_ganador),
        'rmse': float(rmse_ganador)
    }
}

# compress=3 comprime de 178 MB a ~15 MB para pasar por GitHub sin problemas
joblib.dump(paquete_modelo, ruta_export, compress=3)
print(f"✅ Modelo ganador ({nombre_ganador}) exportado exitosamente a: {ruta_export}")
print("Iniciando descarga automática a tu equipo...")
files.download(ruta_export)

✅ Modelo ganador (Random Forest Regressor) exportado exitosamente a: /content/models/modelo_04_aguacate.joblib
Iniciando descarga automática a tu equipo...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Conclusiones del Análisis
1. **Comparativa de Arquitecturas:** La evaluación cuantitativa evidenció que el modelo de ensamble (Random Forest) superó a la Red Neuronal (MLP) en este conjunto de datos. El Bosque Aleatorio logró explicar casi el 91% de la varianza ($R^2$ de 0.9089) con un margen de error promedio (MAE) de apenas $0.08 USD, demostrando empíricamente mayor precisión al mapear esta superficie de precios tabulares.
2. **Factores Económicos Dominantes:** Basado en la extracción de importancia de variables (Gini) del modelo ganador, el tipo de cultivo (`is_organic`) y la estacionalidad (`Month`) se validaron matemáticamente como los principales inductores en la fijación del precio, superando el peso de los volúmenes globales de distribución.
3. **Despliegue en 10N-AI:** El artefacto `.joblib` generado empaqueta el estimador Random Forest junto con sus transformadores e hiperparámetros óptimos, dejándolo estructurado y listo para su consumo directo a través de la API REST del asistente.